# Text Preprocessing for Amazon Reviews
## Sentiment Analysis - Step 2: Data Preprocessing

This notebook preprocesses the extracted Amazon reviews for sentiment analysis.

### Preprocessing Steps:
1. **Convert to lowercase** - Normalize text case
2. **Tokenization** - Split text into words
3. **Remove punctuation** - Clean special characters
4. **Remove stopwords** - Remove common words (and, a, the, etc.)
5. **Lemmatization** - Convert words to base form

### Output:
- Cleaned and preprocessed reviews CSV file
- Before/After comparison
- Statistics on preprocessing impact

In [6]:
# Section 1: Install and Import Libraries
import subprocess
import sys

print("Installing required NLP libraries...\n")

packages = ['pandas', 'nltk']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

# Import libraries
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from datetime import datetime

# Download required NLTK data
print("\nDownloading NLTK resources...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("\n✓ All libraries and resources ready!")

Installing required NLP libraries...

✓ pandas already installed
✓ nltk already installed


✓ All libraries and resources ready!


In [2]:
# Section 2: Load the CSV file
print("Loading reviews from CSV...\n")

# Load the cleaned CSV from the scraper
csv_file = 'amazon_reviews_OnePlus_Nord_5_20260410_101426.csv'

try:
    df = pd.read_csv(csv_file)
    print(f"✓ Loaded {len(df)} reviews from: {csv_file}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 2 reviews:")
    print(df[['review_text', 'rating']].head(2))
except FileNotFoundError:
    print(f"❌ File not found: {csv_file}")
    print(f"\nAvailable CSV files:")
    import os
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    for f in csv_files:
        print(f"  - {f}")

Loading reviews from CSV...

✓ Loaded 100 reviews from: amazon_reviews_OnePlus_Nord_5_20260410_101426.csv

Columns: ['review_text', 'rating', 'reviewer_name', 'review_date']

First 2 reviews:
                                         review_text  rating
0  I’ve been using the OnePlus Nord 5 for a few w...     NaN
1  Been using it over 2 months now and the experi...     NaN


In [3]:
# Section 3: Step 1 - Convert to Lowercase
print("Step 1: Converting text to lowercase...\n")

df['review_lowercase'] = df['review_text'].str.lower()

print("Example:")
print(f"Original: {df['review_text'].iloc[0][:100]}...")
print(f"\nLowercase: {df['review_lowercase'].iloc[0][:100]}...")
print(f"\n✓ All {len(df)} reviews converted to lowercase")

Step 1: Converting text to lowercase...

Example:
Original: I’ve been using the OnePlus Nord 5 for a few weeks now, and honestly, it’s one of the best all‑round...

Lowercase: i’ve been using the oneplus nord 5 for a few weeks now, and honestly, it’s one of the best all‑round...

✓ All 100 reviews converted to lowercase


In [7]:
# Section 4: Step 2 - Tokenization
print("Step 2: Tokenizing text (splitting into words)...\n")

df['tokens'] = df['review_lowercase'].apply(lambda x: word_tokenize(x) if pd.notna(x) else [])

print("Example:")
print(f"Text: {df['review_lowercase'].iloc[0][:80]}...")
print(f"\nTokens (first 20): {df['tokens'].iloc[0][:20]}")
print(f"\nTotal tokens in first review: {len(df['tokens'].iloc[0])}")
print(f"\n✓ All {len(df)} reviews tokenized")

Step 2: Tokenizing text (splitting into words)...

Example:
Text: i’ve been using the oneplus nord 5 for a few weeks now, and honestly, it’s one o...

Tokens (first 20): ['i', '’', 've', 'been', 'using', 'the', 'oneplus', 'nord', '5', 'for', 'a', 'few', 'weeks', 'now', ',', 'and', 'honestly', ',', 'it', '’']

Total tokens in first review: 408

✓ All 100 reviews tokenized


In [8]:
# Section 5: Step 3 - Remove Punctuation
print("Step 3: Removing punctuation...\n")

def remove_punctuation(tokens):
    # Remove tokens that are only punctuation
    return [token for token in tokens if token not in string.punctuation]

df['tokens_no_punct'] = df['tokens'].apply(remove_punctuation)

print("Example:")
print(f"With punctuation (first 20 tokens): {df['tokens'].iloc[0][:20]}")
print(f"\nWithout punctuation (first 20 tokens): {df['tokens_no_punct'].iloc[0][:20]}")
print(f"\nTokens removed: {len(df['tokens'].iloc[0]) - len(df['tokens_no_punct'].iloc[0])}")
print(f"\n✓ Punctuation removed from all {len(df)} reviews")

Step 3: Removing punctuation...

Example:
With punctuation (first 20 tokens): ['i', '’', 've', 'been', 'using', 'the', 'oneplus', 'nord', '5', 'for', 'a', 'few', 'weeks', 'now', ',', 'and', 'honestly', ',', 'it', '’']

Without punctuation (first 20 tokens): ['i', '’', 've', 'been', 'using', 'the', 'oneplus', 'nord', '5', 'for', 'a', 'few', 'weeks', 'now', 'and', 'honestly', 'it', '’', 's', 'one']

Tokens removed: 42

✓ Punctuation removed from all 100 reviews


In [9]:
# Section 6: Step 4 - Remove Stopwords
print("Step 4: Removing stopwords...\n")

stop_words = set(stopwords.words('english'))
print(f"Total English stopwords: {len(stop_words)}")
print(f"Examples: {list(stop_words)[:10]}")

def remove_stopwords(tokens):
    # Remove stopwords from tokens
    return [token for token in tokens if token.lower() not in stop_words]

df['tokens_no_stopwords'] = df['tokens_no_punct'].apply(remove_stopwords)

print(f"\nExample:")
print(f"With stopwords: {df['tokens_no_punct'].iloc[0][:15]}")
print(f"\nWithout stopwords: {df['tokens_no_stopwords'].iloc[0][:15]}")
print(f"\nStopwords removed: {len(df['tokens_no_punct'].iloc[0]) - len(df['tokens_no_stopwords'].iloc[0])}")
print(f"\n✓ Stopwords removed from all {len(df)} reviews")

Step 4: Removing stopwords...

Total English stopwords: 198
Examples: ["isn't", 'being', 'just', 'more', 'same', 'they', "doesn't", 's', 'been', 'their']

Example:
With stopwords: ['i', '’', 've', 'been', 'using', 'the', 'oneplus', 'nord', '5', 'for', 'a', 'few', 'weeks', 'now', 'and']

Without stopwords: ['’', 'using', 'oneplus', 'nord', '5', 'weeks', 'honestly', '’', 'one', 'best', 'all‑round', 'mid‑range', 'phones', '’', 'tried']

Stopwords removed: 128

✓ Stopwords removed from all 100 reviews


In [10]:
# Section 7: Step 5 - Lemmatization
print("Step 5: Lemmatizing words (converting to base form)...\n")

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    # Convert words to their base form
    return [lemmatizer.lemmatize(token) for token in tokens]

df['tokens_lemmatized'] = df['tokens_no_stopwords'].apply(lemmatize_tokens)

print("Examples of lemmatization:")
examples = [
    ('running', 'run'),
    ('better', 'good'),
    ('phones', 'phone'),
    ('was', 'be')
]
for word, base in examples:
    lemmatized = lemmatizer.lemmatize(word)
    print(f"  {word:12} → {lemmatized}")

print(f"\nExample review processing:")
print(f"Before: {df['tokens_no_stopwords'].iloc[0][:10]}")
print(f"After:  {df['tokens_lemmatized'].iloc[0][:10]}")
print(f"\n✓ All {len(df)} reviews lemmatized")

Step 5: Lemmatizing words (converting to base form)...

Examples of lemmatization:
  running      → running
  better       → better
  phones       → phone
  was          → wa

Example review processing:
Before: ['’', 'using', 'oneplus', 'nord', '5', 'weeks', 'honestly', '’', 'one', 'best']
After:  ['’', 'using', 'oneplus', 'nord', '5', 'week', 'honestly', '’', 'one', 'best']

✓ All 100 reviews lemmatized


In [22]:
# Section 8: Step 6 - Remove Special Characters (Encoding Issues)
print("Step 6: Removing special characters and encoding errors...\n")

import unicodedata
import re

def remove_special_chars(text):
    """
    Remove special characters and encoding errors like â€™, â€œ, etc.
    Keep only alphanumeric characters and spaces
    """
    if pd.isna(text):
        return text
    # Normalize unicode characters
    text = unicodedata.normalize('NFKD', str(text))
    # Remove non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

# First clean the original review_text to remove encoding errors
print("Cleaning original review text...")
df['review_text'] = df['review_text'].apply(remove_special_chars)

# Then create cleaned text from tokens and clean that too
df['cleaned_text'] = df['tokens_lemmatized'].apply(lambda tokens: ' '.join(tokens))
df['cleaned_text'] = df['cleaned_text'].apply(remove_special_chars)

print("Examples of special character removal:")
print("  - Special quote characters -> removed")
print("  - Double quote characters -> removed")
print("  - Right quote characters -> removed")
print("  - Em dash characters -> removed")

print(f"\nExample:")
sample_text = "best all round phones amazing"
print(f"Before: 'best [special-char] all round phones [em-dash] amazing'")
print(f"After:  '{remove_special_chars(sample_text)}'")

print(f"\n✓ Special characters removed from all {len(df)} reviews")

# Create final dataframe with original and cleaned data
df_final = df[['review_text', 'rating', 'reviewer_name', 'review_date', 'cleaned_text']].copy()

print(f"\nFinal DataFrame shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\n" + "="*80)
print("BEFORE AND AFTER COMPARISON (Full Pipeline)")
print("="*80)

sample_idx = 0
print(f"\nReview #{sample_idx + 1}:")
print(f"\nORIGINAL TEXT (WITH SPECIAL CHARS REMOVED):")
print(f"{df_final['review_text'].iloc[sample_idx]}")
print(f"\nCLEANED TEXT (After all 6 steps):")
print(f"{df_final['cleaned_text'].iloc[sample_idx]}")
print(f"\nStats for this review:")
print(f"  Original length: {len(df_final['review_text'].iloc[sample_idx])} characters")
print(f"  Original tokens: {len(df['tokens'].iloc[sample_idx])} tokens")
print(f"  Final tokens: {len(df['tokens_lemmatized'].iloc[sample_idx])} tokens")
print(f"  Reduction: {len(df['tokens'].iloc[sample_idx]) - len(df['tokens_lemmatized'].iloc[sample_idx])} tokens removed")

print(f"\n✓ Final DataFrame created with {len(df_final)} reviews (all special characters removed from ALL columns)")

Step 6: Removing special characters and encoding errors...

Cleaning original review text...
Examples of special character removal:
  - Special quote characters -> removed
  - Double quote characters -> removed
  - Right quote characters -> removed
  - Em dash characters -> removed

Example:
Before: 'best [special-char] all round phones [em-dash] amazing'
After:  'best all round phones amazing'

✓ Special characters removed from all 100 reviews

Final DataFrame shape: (100, 5)
Columns: ['review_text', 'rating', 'reviewer_name', 'review_date', 'cleaned_text']

BEFORE AND AFTER COMPARISON (Full Pipeline)

Review #1:

ORIGINAL TEXT (WITH SPECIAL CHARS REMOVED):
Ive been using the OnePlus Nord 5 for a few weeks now, and honestly, its one of the best allround midrange phones Ive tried in a long time. The Snapdragon 8s Gen 3 delivers incredible speed multitasking and gaming (especially BGMI and COD Mobile) feel supersmooth thanks to the 144 Hz AMOLED display. The 6.83inch screen is bright an

In [23]:
# Section 9: Save Preprocessed Data
print("Saving preprocessed data (with special characters removed)...\n")

# Save with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"amazon_reviews_preprocessed_clean_{timestamp}.csv"

df_final.to_csv(output_filename, index=False, encoding='utf-8')

print(f"✓ Preprocessed data saved to: {output_filename}")
print(f"\nFile details:")
print(f"  - Number of reviews: {len(df_final)}")
print(f"  - Columns: {', '.join(df_final.columns)}")
print(f"  - All special characters removed ✓")
print(f"  - Ready for TF-IDF and bag-of-words analysis ✓")

Saving preprocessed data (with special characters removed)...

✓ Preprocessed data saved to: amazon_reviews_preprocessed_clean_20260410_105305.csv

File details:
  - Number of reviews: 100
  - Columns: review_text, rating, reviewer_name, review_date, cleaned_text
  - All special characters removed ✓
  - Ready for TF-IDF and bag-of-words analysis ✓


In [24]:
# Section 10: Preprocessing Statistics and Summary
print("\n" + "="*80)
print("PREPROCESSING STATISTICS SUMMARY")
print("="*80)

# Calculate statistics
original_avg_length = df['review_text'].str.len().mean()
cleaned_avg_length = df['cleaned_text'].str.len().mean()

original_avg_tokens = df['tokens'].apply(len).mean()
cleaned_avg_tokens = df['tokens_lemmatized'].apply(len).mean()

print(f"\n📊 TEXT LENGTH STATISTICS:")
print(f"  Original average length: {original_avg_length:.0f} characters")
print(f"  Cleaned average length:  {cleaned_avg_length:.0f} characters")
print(f"  Reduction: {((original_avg_length - cleaned_avg_length) / original_avg_length * 100):.1f}%")

print(f"\n🔤 TOKEN STATISTICS:")
print(f"  Original average tokens: {original_avg_tokens:.0f} tokens")
print(f"  Cleaned average tokens:  {cleaned_avg_tokens:.0f} tokens")
print(f"  Reduction: {((original_avg_tokens - cleaned_avg_tokens) / original_avg_tokens * 100):.1f}%")

print(f"\n⭐ RATING DISTRIBUTION:")
rating_dist = df_final['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    percentage = (count / len(df_final)) * 100
    bar = '█' * int(percentage / 2)
    print(f"  {rating} stars: {count:3d} reviews ({percentage:5.1f}%) {bar}")

print(f"\n📝 DATA QUALITY:")
print(f"  Total reviews processed: {len(df_final)}")
print(f"  Reviews with valid text: {df_final['cleaned_text'].notna().sum()}")
print(f"  Reviews with ratings: {df_final['rating'].notna().sum()}")
print(f"  Null values: {df_final.isnull().sum().sum()}")

print(f"\n✓ Preprocessing completed successfully!")


PREPROCESSING STATISTICS SUMMARY

📊 TEXT LENGTH STATISTICS:
  Original average length: 844 characters
  Cleaned average length:  603 characters
  Reduction: 28.6%

🔤 TOKEN STATISTICS:
  Original average tokens: 165 tokens
  Cleaned average tokens:  91 tokens
  Reduction: 44.8%

⭐ RATING DISTRIBUTION:

📝 DATA QUALITY:
  Total reviews processed: 100
  Reviews with valid text: 100
  Reviews with ratings: 0
  Null values: 200

✓ Preprocessing completed successfully!


In [25]:
# Section 11: Display Sample Preprocessed Reviews
print("\n" + "="*80)
print("SAMPLE PREPROCESSED REVIEWS")
print("="*80)

for idx in range(min(3, len(df_final))):
    print(f"\n{'─'*80}")
    print(f"Review #{idx + 1}")
    print(f"Rating: {df_final['rating'].iloc[idx]} | Reviewer: {df_final['reviewer_name'].iloc[idx]}")
    print(f"\n📝 ORIGINAL TEXT:")
    print(f"{df_final['review_text'].iloc[idx][:300]}..." if len(df_final['review_text'].iloc[idx]) > 300 else df_final['review_text'].iloc[idx])
    print(f"\n✨ CLEANED TEXT:")
    print(f"{df_final['cleaned_text'].iloc[idx][:300]}..." if len(df_final['cleaned_text'].iloc[idx]) > 300 else df_final['cleaned_text'].iloc[idx])
    print(f"\n📊 Metrics:")
    print(f"  Original: {len(df['tokens'].iloc[idx])} tokens")
    print(f"  Cleaned:  {len(df['tokens_lemmatized'].iloc[idx])} tokens")
    print(f"  Removed:  {len(df['tokens'].iloc[idx]) - len(df['tokens_lemmatized'].iloc[idx])} tokens ({((len(df['tokens'].iloc[idx]) - len(df['tokens_lemmatized'].iloc[idx])) / len(df['tokens'].iloc[idx]) * 100):.1f}%)")

print(f"\n{'─'*80}")
print(f"\n✅ Preprocessing complete! Data ready for sentiment analysis.")


SAMPLE PREPROCESSED REVIEWS

────────────────────────────────────────────────────────────────────────────────
Review #1
Rating: nan | Reviewer: Anonymous

📝 ORIGINAL TEXT:
Ive been using the OnePlus Nord 5 for a few weeks now, and honestly, its one of the best allround midrange phones Ive tried in a long time. The Snapdragon 8s Gen 3 delivers incredible speed multitasking and gaming (especially BGMI and COD Mobile) feel supersmooth thanks to the 144 Hz AMOLED display....

✨ CLEANED TEXT:
using oneplus nord 5 week honestly one best allround midrange phone tried long time snapdragon 8 gen 3 delivers incredible speed multitasking gaming especially bgmi cod mobile feel supersmooth thanks 144 hz amoled display 6.83inch screen bright crisp easily visible sunlight 3000 hz touch response ma...

📊 Metrics:
  Original: 408 tokens
  Cleaned:  238 tokens
  Removed:  170 tokens (41.7%)

────────────────────────────────────────────────────────────────────────────────
Review #2
Rating: nan | Reviewe